#### Scraping arabic tech news

we collect arabic texts about one topic (technology) from 3 websites: aitnews, arabhardware and aljazeera (tech section). the plan is simple:
1. get the list of article urls from the sitemap of each site (an xml file that lists the articles)
2. open each article and keep the title and the text
3. save everything in a csv file

note: this notebook does not need a GPU, so run it with a local kernel (not colab) to get the csv in the repository

In [1]:
import os, time, random
from datetime import date, timedelta

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

random.seed(0)

# some websites refuse requests without a browser name
headers = {"User-Agent": "Mozilla/5.0"}

# number of articles we want from each website
N_PER_SITE = 1500

os.makedirs("../dataset", exist_ok=True)

##### step 1: read a sitemap

In [2]:
import warnings
warnings.filterwarnings("ignore")  # bs4 warns when we read xml with html.parser

def get_urls(sitemap_url):
    # a sitemap is an xml file, every article url is inside a <loc> tag
    response = requests.get(sitemap_url, headers=headers, timeout=20)
    soup = BeautifulSoup(response.content, "html.parser")
    return [loc.text.strip() for loc in soup.find_all("loc")]

# test
test_urls = get_urls("https://aitnews.com/post-sitemap80.xml")
print(len(test_urls))
print(test_urls[:3])

532
['https://aitnews.com/2026/05/19/%d8%a5%d9%83%d8%b3-%d8%aa%d9%81%d8%b1%d8%b6-%d9%82%d9%8a%d9%88%d8%af%d9%8b%d8%a7-%d8%b5%d8%a7%d8%b1%d9%85%d8%a9-%d8%b9%d9%84%d9%89-%d8%a7%d9%84%d8%ad%d8%b3%d8%a7%d8%a8%d8%a7%d8%aa-%d8%a7%d9%84%d9%85/', 'https://aitnews.com/2026/05/19/%d8%a8%d8%b9%d8%af-%d8%b3%d9%86%d9%88%d8%a7%d8%aa-%d9%85%d9%86-%d8%a7%d9%84%d8%b4%d9%83%d8%a7%d9%88%d9%89-%d9%85%d8%a7%d9%8a%d9%83%d8%b1%d9%88%d8%b3%d9%88%d9%81%d8%aa-%d8%aa%d9%83%d8%aa%d8%b4%d9%81/', 'https://aitnews.com/2026/05/19/%d9%82%d8%af%d8%b1%d8%a7%d8%aa-%d8%ae%d8%a7%d8%b1%d9%82%d8%a9-%d8%aa%d9%82%d9%84%d9%82-%d8%a7%d9%84%d8%a8%d9%86%d9%88%d9%83-%d9%86%d9%85%d9%88%d8%b0%d8%ac-claude-mythos-%d8%aa%d8%ad%d8%aa-%d8%b1/']


##### step 2: read one article

In [3]:
def scrape_article(url, selector):
    # returns (title, text) or None if something is wrong
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.content, "html.parser")
        title = soup.find("h1")
        body = soup.select_one(selector)   # the block that contains the article text
        if title is None or body is None:
            return None

        # the text is the paragraphs inside the body block
        text = " ".join(p.get_text(" ", strip=True) for p in body.find_all("p"))
        return title.get_text(strip=True), text
    except Exception:
        return None

# the css selector of the article text is different for each website
selectors = {
    "aitnews": "div.entry-content",
    "arabhardware": "div.post-content",
    "aljazeera": "div.wysiwyg",
}

# test on one article of aitnews
title, text = scrape_article(test_urls[-1], selectors["aitnews"])
print(title)
print(len(text))
print(text[:200])

علي بابا تطلق Qwen4-Max بقدرات متعددة الوسائط وسياق مليون رمز
2097
أطلقت شركة علي بابا اليوم نموذج الذكاء الاصطناعي Qwen4-Max ، مع دعم فهم النصوص والصور والفيديو ونافذة سياق تصل إلى مليون رمز، مستهدفةً مهام البرمجة والإنتاجية المكتبية والأعمال التي تتطلب تنفيذ خطوات 


##### step 3: collect the urls of each website

the sitemaps are ordered from old to new, so we take the recent ones. then we pick random articles

In [4]:
# aitnews: the 2 last sitemaps (recent articles)
ait_urls = get_urls("https://aitnews.com/post-sitemap80.xml") + get_urls("https://aitnews.com/post-sitemap79.xml")
ait_urls = random.sample(ait_urls, min(N_PER_SITE, len(ait_urls)))

# arabhardware: sitemap of articles and sitemap of reviews (hardware / tech)
ah_urls = get_urls("https://arabhardware.net/sitemap/articles-sitemap1.xml") + get_urls("https://arabhardware.net/sitemap/reviews-sitemap1.xml")
ah_urls = random.sample(ah_urls, min(N_PER_SITE, len(ah_urls)))

# aljazeera: one sitemap per day, we keep only the tech and ebusiness sections
aj_urls = []
day = date.today()
while len(aj_urls) < N_PER_SITE:
    sitemap = f"https://www.aljazeera.net/sitemap.xml?yyyy={day.year}&mm={day.month}&dd={day.day}"
    urls = get_urls(sitemap)
    aj_urls += [u for u in urls if "/tech/" in u or "/ebusiness/" in u]
    day = day - timedelta(days=1)
aj_urls = aj_urls[:N_PER_SITE]

print(len(ait_urls), len(ah_urls), len(aj_urls))

1500 1500 1500


##### step 4: scrape all the articles

we wait a bit between two requests to be polite with the websites. each website is saved in its own csv, so if something stops we do not lose everything

In [5]:
def scrape_site(name, urls):
    path = f"../dataset/raw_{name}.csv"

    # if a previous run saved some articles, we keep them and we scrape only the new urls
    rows = pd.read_csv(path).to_dict("records") if os.path.exists(path) else []
    done = {row["url"] for row in rows}
    todo = [u for u in urls if u not in done]
    print(name, "->", len(done), "articles already saved,", len(todo), "to scrape")

    for i, url in enumerate(tqdm(todo, desc=name)):
        result = scrape_article(url, selectors[name])
        if result is not None:
            title, text = result
            rows.append({"title": title, "text": text, "source": name, "url": url})
        time.sleep(0.5)

        # save every 100 articles, if the notebook stops we do not lose everything
        if (i + 1) % 100 == 0:
            pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8")

    df = pd.DataFrame(rows)
    df.to_csv(path, index=False, encoding="utf-8")
    print(name, "->", len(df), "articles")
    return df

df_ait = scrape_site("aitnews", ait_urls)

aitnews -> 300 articles already saved, 1201 to scrape


aitnews:   0%|          | 0/1201 [00:00<?, ?it/s]

aitnews:   0%|          | 1/1201 [00:00<16:34,  1.21it/s]

aitnews:   0%|          | 2/1201 [00:01<16:14,  1.23it/s]

aitnews:   0%|          | 3/1201 [00:02<15:57,  1.25it/s]

aitnews:   0%|          | 4/1201 [00:03<15:54,  1.25it/s]

... (1,195 outputs left out)

aitnews: 100%|█████████▉| 1200/1201 [15:48<00:00,  1.23it/s]

aitnews: 100%|██████████| 1201/1201 [15:49<00:00,  1.28it/s]

aitnews: 100%|██████████| 1201/1201 [15:49<00:00,  1.27it/s]

aitnews -> 1501 articles


In [6]:
df_ah = scrape_site("arabhardware", ah_urls)

arabhardware -> 296 articles already saved, 1271 to scrape


arabhardware:   0%|          | 0/1271 [00:00<?, ?it/s]

arabhardware:   0%|          | 1/1271 [00:01<33:22,  1.58s/it]

arabhardware:   0%|          | 2/1271 [00:03<33:19,  1.58s/it]

arabhardware:   0%|          | 3/1271 [00:04<31:24,  1.49s/it]

arabhardware:   0%|          | 4/1271 [00:05<30:01,  1.42s/it]

... (1,265 outputs left out)

arabhardware: 100%|█████████▉| 1270/1271 [29:37<00:01,  1.40s/it]

arabhardware: 100%|██████████| 1271/1271 [29:38<00:00,  1.38s/it]

arabhardware: 100%|██████████| 1271/1271 [29:38<00:00,  1.40s/it]

arabhardware -> 1564 articles


In [7]:
df_aj = scrape_site("aljazeera", aj_urls)

aljazeera -> 300 articles already saved, 1200 to scrape


aljazeera:   0%|          | 0/1200 [00:00<?, ?it/s]

aljazeera:   0%|          | 1/1200 [00:01<29:42,  1.49s/it]

aljazeera:   0%|          | 2/1200 [00:02<21:56,  1.10s/it]

aljazeera:   0%|          | 3/1200 [00:03<26:44,  1.34s/it]

aljazeera:   0%|          | 4/1200 [00:05<27:40,  1.39s/it]

... (1,194 outputs left out)

aljazeera: 100%|█████████▉| 1199/1200 [27:56<00:00,  1.36it/s]

aljazeera: 100%|██████████| 1200/1200 [27:57<00:00,  1.31it/s]

aljazeera: 100%|██████████| 1200/1200 [27:57<00:00,  1.40s/it]

aljazeera -> 1500 articles


##### step 5: merge and clean

In [8]:
df = pd.concat([df_ait, df_ah, df_aj], ignore_index=True)

# remove duplicates and texts that are too short
df = df.drop_duplicates(subset="url")
df = df.drop_duplicates(subset="text")
df = df[df["text"].str.len() >= 200].reset_index(drop=True)

df.to_csv("../dataset/arabic_texts.csv", index=False, encoding="utf-8")

print(df.shape)
print(df["source"].value_counts())
print(df["text"].str.len().describe())
df.head()

(4555, 4)
source
arabhardware    1563
aljazeera       1500
aitnews         1492
Name: count, dtype: int64
count     4555.000000
mean      5009.132821
std       4108.556300
min        236.000000
25%       2211.000000
50%       3654.000000
75%       6562.000000
max      45426.000000
Name: text, dtype: float64


,title,text,source,url
0,سيري تُسقِط قائدها.. آبل تعلن رحيل رئيس الذكاء...,أعلنت شركة آبل مغادرة رئيس قسم الذكاء الاصطناع...,aitnews,https://aitnews.com/2025/12/02/%d8%b3%d9%8a%d8...
1,مايكروسوفت تطلق قسمًا جديدًا للسِمات في متجر و...,أعلنت مايكروسوفت إطلاق قسم جديد للسِمات (Theme...,aitnews,https://aitnews.com/2025/12/12/%d9%85%d8%a7%d9...
2,هاتف ترامب تحت المجهر.. الكشف عن تفاصيل مُحرجة...,كشف موقع iFixit المتخصص في تفكيك الأجهزة الإلك...,aitnews,https://aitnews.com/2026/06/13/%d9%87%d8%a7%d8...
3,بوني إيه آي وتويوتا تطلقان سيارات أجرة bZ4X بل...,أعلنت بوني إيه آي (Pony.ai) وتويوتا اليوم بدء ...,aitnews,https://aitnews.com/2026/11/16/pony-ai-toyota-...
4,فوجي فيلم الشرق الأوسط تطلق حزمة حلول لطب القل...,أطلقت شركة فوجي فيلم الشرق الأوسط حزمة حلول مت...,aitnews,https://aitnews.com/2026/02/04/%d9%81%d9%88%d8...
